In [4]:
# %%
import sys
import os
sys.path.extend(["..\\"])
import math as m
import pandas as pd
from  Shared.NIBSData2 import *
import seaborn.objects as so
import seaborn as sns
import matplotlib.pyplot as plt
# import patchworklib as pw
import numpy as np
from scipy import stats
import itertools
import geopandas as gpd 
# import PostDoc.db_clients.mssql_db_client as mssql  
# import PostDoc.Plotting.squarify as squarify
import copy
# import docx 
# from docx.shared import Cm 
# from PostDoc.Plotting.PlottingFunctions import *
# from plotnine import ggplot, geom_point, aes, stat_smooth, facet_wrap, scale_color_discrete, labs
import plotnine as p9

chart_dir = r"./Graphs/residue_burning"
if not os.path.exists(chart_dir):
       os.makedirs(chart_dir)
       
       
nibs = NIBSData(data_dir=r"./Data" , 
                chart_dir=r"./Graphs/residue_burning",)

 
# print("Loading national CH4 summary data...")
# try:
#     ch4_sql_national = f"""select * 
#     from read_parquet('{path.join(nibs.data_dir,'vwG_national_ch4_summary_all_models.parquet')}')"""
#     ch4_df_national = nibs.load_table_data(ch4_sql_national)
#     ch4_df_national
# except Exception as e:
#     print("Error loading national CH4 summary data:", e)


# #load india national boundary as map background
# india_sql = "SELECT geog.STAsBinary() as geog FROM [dbo].[national_boundaries]"
# india = load_map_data(db_client, india_sql)
# base_map = plot_map(None, india, None, 'base map', '')
# base_map

# # db_conn = {'server': '.\\npongo22', 'database': 'india_cost_of_cultivation_ghg_results_v1'}
# # db_client = mssql.SqlServerClient(db_conn) 

# db_conn_input = {'server': '.\\npongo22', 'database': 'india_agriculture_census_ghg_results_v1'}
# db_client_input = mssql.SqlServerClient(db_conn_input)


print('Python %s on %s' % (sys.version, sys.platform))


Python 3.13.14 (tags/v3.13.14:fd17997, Jun 10 2026, 13:03:48) [MSC v.1944 64 bit (AMD64)] on win32


In [ ]:
farm_size_residue_burning_co2e_sql ="""
select *
from vwG_national_farm_size_residue_burning_co2e_summary 
"""
farm_size_residue_burning_co2e_df = load_table_data(db_client_input, farm_size_residue_burning_co2e_sql)
farm_size_residue_burning_co2e_df.head()

In [ ]:
farm_size_residue_burning_co2e_df = farm_size_residue_burning_co2e_df.sort_values(by=['gwp_time_period', 'farm_size'], ascending=[True,True])
print(list(farm_size_residue_burning_co2e_df['farm_size']))
print({i:i for i in farm_size_residue_burning_co2e_df['farm_size']})   

In [ ]:
farm_size_residue_burning_co2e_df['farm_size'] = pd.Categorical(farm_size_residue_burning_co2e_df['farm_size']
                                    , categories=['LARGE (10 AND ABOVE)', 'MEDIUM (4.0 - 9.99)', 'SEMI-MEDIUM (2.0 - 3.99)', 'SMALL (1.0 - 1.99)', 'MARGINAL (BELOW 1.0)'], ordered=True)
farm_size_residue_burning_co2e_df['farm_size'] = farm_size_residue_burning_co2e_df['farm_size'].cat.rename_categories({'LARGE (10 AND ABOVE)': '≥10 Ha', 
                                                                                               'MEDIUM (4.0 - 9.99)': '4.0 - 9.99 Ha', 
                                                                                               'SEMI-MEDIUM (2.0 - 3.99)': '2.0 - 3.99 Ha', 
                                                                                               'SMALL (1.0 - 1.99)': '1.0 - 1.99 Ha', 
                                                                                               'MARGINAL (BELOW 1.0)': '<1.0 Ha'})

farm_size_residue_burning_co2e_df['mean_kg_co2e_ha_min'] = farm_size_residue_burning_co2e_df.apply(
    lambda row: 0 if row['mean_kg_co2e_ha'] - row['sd_kg_co2e_ha'] < 0 else row['mean_kg_co2e_ha'] - row['sd_kg_co2e_ha'],
    axis=1
)

gwp_label={20:'$GWP_{20}$', 100:'$GWP_{100}$'}
farm_size_residue_burning_co2e_df['gwp_label'] = farm_size_residue_burning_co2e_df['gwp_time_period'].map(gwp_label)

farm_size_residue_burning_co2e_df['gwp_label'] = pd.Categorical(farm_size_residue_burning_co2e_df['gwp_label']
                                    , categories=list(farm_size_residue_burning_co2e_df['gwp_label'].unique()), ordered=True)
farm_size_residue_burning_co2e_df.head()

In [ ]:
p = (ggplot()
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_col(farm_size_residue_burning_co2e_df,aes(x='farm_size', y='mean_kg_co2e_ha',fill='farm_size'))
            + geom_errorbar(farm_size_residue_burning_co2e_df, aes(x='farm_size', ymin='mean_kg_co2e_ha_min', ymax='mean_kg_co2e_ha + sd_kg_co2e_ha'), width=0.2)
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Crop Residue Burning Emissions', x="Farm Size", y="$CO_2e_{100}\ Kg\ Ha^{-1}\ Year^{-1}$")
            # + scale_y_log10()
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=90, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
             + facet_wrap('~gwp_label')
            )
print(p)
p.save(filename=f"national_farm_size_residue_burning_mean_kg_co2e_ha_14x25.png", path=chart_dir,height=14, width=25, units='cm', dpi=92)
p.save(filename=f"national_farm_size_residue_burning_mean_kg_co2e_ha_14x12.png", path=chart_dir,height=14, width=12, units='cm', dpi=92)

In [ ]:
# farm_size_residue_burning_co2e_df = farm_size_residue_burning_co2e_df.sort_values(by=[ 'mean_kg_co2e_kg_yield'], ascending=[True])
# farm_size_residue_burning_co2e_df['farm_size'] = pd.Categorical(farm_size_residue_burning_co2e_df['farm_size']
#                                     , categories=list(farm_size_residue_burning_co2e_df['farm_size'].unique())
# , ordered=True)

farm_size_residue_burning_co2e_df['mean_kg_co2e_kg_yield_min'] = farm_size_residue_burning_co2e_df.apply(
    lambda row: 0 if row['mean_kg_co2e_kg_yield'] - row['sd_kg_co2e_kg_yield'] < 0 else row['mean_kg_co2e_kg_yield'] - row['sd_kg_co2e_kg_yield'],
    axis=1
)

In [ ]:
p = (ggplot()
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_col(farm_size_residue_burning_co2e_df,aes(x='farm_size', y='mean_kg_co2e_kg_yield',fill='farm_size'))
            + geom_errorbar(farm_size_residue_burning_co2e_df, aes(x='farm_size', ymin='mean_kg_co2e_kg_yield_min', ymax='mean_kg_co2e_kg_yield + sd_kg_co2e_kg_yield'), width=0.2)
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Crop Residue Burning Emissions by Yield', x="Farm Size", y="$CO_2e_{100}\ Kg\ Kg\ Yield^{-1}\ Year^{-1}$")
            # + scale_y_log10()
            + guides(fill=None) 
            + theme(axis_text_x=element_text(angle=90, va="top", ha="center", size=10), plot_title=element_text(ha='center', size=14))
             + facet_wrap('~gwp_label')
            )
print(p)
p.save(filename=f"national_farm_size_mean_kg_co2e_kg_yield_14x25.png", path=chart_dir,height=14, width=25, units='cm', dpi=92)

In [ ]:
# farm_size_residue_burning_co2e_df = farm_size_residue_burning_co2e_df.sort_values(by=[ 'mean_total_t_co2e'], ascending=[True])
# farm_size_residue_burning_co2e_df['farm_size'] = pd.Categorical(farm_size_residue_burning_co2e_df['farm_size']
#                                     , categories=list(farm_size_residue_burning_co2e_df['farm_size'].unique())
# , ordered=True)

farm_size_residue_burning_co2e_df['mean_total_t_co2e_min'] = farm_size_residue_burning_co2e_df.apply(
    lambda row: 0 if row['mean_total_t_co2e'] - row['sd_total_t_co2e'] < 0 else row['mean_total_t_co2e'] - row['sd_total_t_co2e'],
    axis=1
)

In [ ]:
p = (ggplot()
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_col(farm_size_residue_burning_co2e_df,aes(x='farm_size', y='mean_total_t_co2e/1000000',fill='farm_size'))
            + geom_errorbar(farm_size_residue_burning_co2e_df, aes(x='farm_size', ymin='mean_total_t_co2e_min/1000000', ymax='mean_total_t_co2e/1000000 + sd_total_t_co2e/1000000'), width=0.2)
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Total Crop Residue Burning Emissions', x="Farm Size", y="$CO_2e_{100}\ Mt^{-1}\ Year^{-1}$")
            #+ scale_y_log10()
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=90, va="top", ha="center", size=10),plot_title=element_text(ha='center', size=14))
             + facet_wrap('~gwp_label')
            )
print(p)
p.save(filename=f"national_farm_size_residue_farm_size_mean_total_t_co2e_14x25.png", path=chart_dir,height=14, width=25, units='cm', dpi=92)
p.save(filename=f"national_farm_size_residue_farm_size_mean_total_t_co2e_14x12.png", path=chart_dir,height=14, width=12, units='cm', dpi=92)

g = (p  + labs(title='2016-17 Total Crop Residue Burning Emissions', x="Farm Size", y="log($CO_2e_{100}\ Mt^{-1}\ Year^{-1}$)")
            + scale_y_log10())

g.save(filename=f"national_farm_size_farm_size_residue_mean_total_t_co2e_log_14x25.png", path=chart_dir,height=14, width=25, units='cm', dpi=92)
g.save(filename=f"national_farm_size_farm_size_residue_mean_total_t_co2e_log_14x12.png", path=chart_dir,height=14, width=12, units='cm', dpi=92)

print(g)

In [ ]:
import numpy as np
import math as m
import pandas as pd
from scipy import stats
import itertools
import geopandas as gpd 
import PostDoc.db_clients.mssql_db_client as mssql  
import PostDoc.Plotting.squarify as squarify
import copy
import docx 
from docx.shared import Cm 
from PostDoc.Plotting.PlottingFunctions import *
from plotnine import ggplot, geom_point, aes, stat_smooth, facet_wrap, scale_color_discrete, labs


chart_dir = r"E:\npongo Dropbox\benjamin clark\CIL\Products\EDF20240913"
if not os.path.exists(chart_dir):
       os.makedirs(chart_dir)

print('Python %s on %s' % (sys.version, sys.platform))
#load india national boundary as map background
india_sql = "SELECT geog.STAsBinary() as geog FROM [dbo].[national_boundaries]"
india = load_map_data(db_client, india_sql)
base_map = plot_map(None, india, None, 'base map', '')
base_map

state_sql = "select * from vwM_india_states option(maxrecursion 0)"
india_states = load_map_data(db_client, state_sql)



# db_conn = {'server': '.\\npongo22', 'database': 'india_cost_of_cultivation_ghg_results_v1'}
# db_client = mssql.SqlServerClient(db_conn) 

db_conn_input = {'server': '.\\npongo22', 'database': 'india_agriculture_census_ghg_results_v1'}
db_client_input = mssql.SqlServerClient(db_conn_input)

In [ ]:
district_sql = """select * from district_boundaries"""
district_df = load_map_data(db_client, district_sql)
district_df.head()

In [ ]:
district_sql = """select district_name, geog.STAsBinary() as geog from district_boundaries"""
district_df = load_map_data(db_client, district_sql)
district_df.head()

In [ ]:
p =  plot_map(None, district_df, None, 'base map', '')

In [ ]:
p =  plot_map(None, district_df, None, 'base map', '')
p

In [ ]:
district_climate_sql = """select  geog.STAsBinary() as geog from district_climate"""
district_climate_df = load_map_data(db_client, district_climate_sql)
district_climate_df.head()

In [ ]:
district_sql = """select district_name, geog.STAsBinary() as geog from district_boundaries"""
district_df = load_map_data(db_client_input, district_sql)
district_df.head()

In [ ]:
p =  plot_map(None, district_df, None, 'base map', '')
p

In [ ]:
district_climate_sql = """select  geog.STAsBinary() as geog from district_climate"""
district_climate_df = load_map_data(db_client_input, district_climate_sql)
district_climate_df.head()

In [ ]:
p =  plot_map(None, district_climate_df, None, 'base map', '')

In [ ]:
p =  plot_map(None, district_climate_df, None, 'base map', '')
p

In [ ]:
district_climate_sql = """select  geog.STAsBinary() as geog from district_climate_temp"""
district_climate_df = load_map_data(db_client_input, district_climate_sql)
district_climate_df.head()

In [ ]:
p =  plot_map(None, district_climate_df, None, 'base map', '')
p

In [ ]:
district_climate_sql = """select  geog.STAsBinary() as geog from district_climate_temp"""
district_climate_df = load_map_data(db_client_input, district_climate_sql)
district_climate_df.head()

In [ ]:
p =  plot_map(None, district_climate_df, None, 'base map', '')
p

In [ ]:
rice_wheat_sql = """select  * from vwR_rice_wheat_n_balance"""
rice_wheat_df = load_table_data(db_client_input, rice_wheat_sql)
rice_wheat_df.head()

In [ ]:
p = (ggplot(national_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$CO_2e\ Mt\ Year^{-1}$")
            # + scale_y_log10()
             +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$CO_2e\ Mt\ Year^{-1}$")
            # + scale_y_log10()
             +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$CO_2e\ Mt\ Year^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
rice_wheat_sql = """select  * from vwR_rice_wheat_n_balance"""
rice_wheat_df = load_table_data(db_client_input, rice_wheat_sql)
rice_wheat_df.head()

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$CO_2e\ Mt\ Year^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$Yield\ Ton\ Ha^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
g

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
)
g

In [ ]:
rice_wheat_sql = """select  * from vwR_rice_wheat_n_balance"""
rice_wheat_df = load_table_data(db_client_input, rice_wheat_sql)
rice_wheat_df.head()

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$Yield\ Ton\ Ha^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='apy_crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$Yield\ Ton\ Ha^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            + guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='apy_crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$Yield\ Ton\ Ha^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            #+ guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
      + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotation_id'))
    
)
g

In [ ]:
rice_wheat_sql = """select  * from vwR_rice_wheat_n_balance"""
rice_wheat_df = load_table_data(db_client_input, rice_wheat_sql)
rice_wheat_df.head()

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
      + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotion_id'))
    
)
g

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
      + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotion_id', color='geog_checksum'))
    
)
g

In [ ]:
rice_wheat_sql = """select  * from vwR_rice_wheat_n_balance"""
rice_wheat_df = load_table_data(db_client_input, rice_wheat_sql)
rice_wheat_df.head()

In [ ]:
rice_wheat_sql = """select  * from vwR_rice_wheat_n_balance"""
rice_wheat_df = load_table_data(db_client_input, rice_wheat_sql)
rice_wheat_df.head()

In [ ]:
p = (ggplot(rice_wheat_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='apy_crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$Yield\ Ton\ Ha^{-1}$")
            # + scale_y_log10()
            #  +scale_y_continuous(limits=(0, 375))
            #+ guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
# p.save(filename=f"national_ghg_15x18.png", path=chart_dir,height=12, width=15, units='cm', dpi=92)

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
      + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotion_id', color='geog_checksum'))
    
)
g

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
      + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotion_id', color='district_name'))
    
)
g

In [ ]:
g = (p + scale_y_continuous(limits=(0, 2))
    + scale_x_continuous(limits=(200, 600))
      + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotation_id', color='district_name'))
    
)
g

In [ ]:
filter_df = rice_wheat_df[(rice_wheat_df['mean_total_n_balance_n_kg_ha'] > 200 and rice_wheat_df['mean_yield_t_ha'] < 2)]
p = (ggplot(filter_df)
            # + geom_violin(filter_df,aes(x='dataset', y='kg_inorganic_n_ha'),draw_quantiles=[0.25, 0.5, 0.75])
            + geom_point(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha',fill='apy_crop'))
        #     + geom_errorbar(aes(x='label_simple', ymin='mean_co2e_Mt - sd_co2e_Mt', ymax='mean_co2e_Mt + sd_co2e_Mt'), width=0.2)
        #   + geom_text(aes(x='label_simple', y='mean_co2e_Mt', label='mean_co2e_Mt'), 
        #          va='bottom', ha='center', size=8, format_string='{:.2f}')
    
            # + geom_jitter(filter_df, aes(x='dataset', y='kg_inorganic_n_ha'), color='blue', alpha=.01, size=.001, width=0.3)
             + labs(title='2016-17 Cropping GHG Emissions', x="Emission Type", y="$Yield\ Ton\ Ha^{-1}$")
            # + scale_y_log10()
            + scale_y_continuous(limits=(0, 2))
            + scale_x_continuous(limits=(200, 600))
            + geom_line(aes(x='mean_total_n_balance_n_kg_ha', y='mean_yield_t_ha', group='rotation_id', color='district_name'))
    
            #  +scale_y_continuous(limits=(0, 375))
            #+ guides(fill=None) 
             + theme(axis_text_x=element_text(angle=0, va="top", ha="center", size=10),
             plot_title=element_text(ha='center', size=14))
            # + facet_wrap('~gwp_label')
            )
print(p)
g = (p 
)
g